# TB Control · AI Inference Server (Kaggle)

Запускает vLLM (LLM + Vision) + YOLO + Cloudflare Tunnel на 2× T4.
Цель: публичный HTTPS endpoint для нашего Next.js приложения.

**Перед запуском:**
1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **On**
3. Run All (займёт ~15 мин: установка + загрузка моделей + retrain YOLO + запуск)

После загрузки последняя ячейка выведет публичный URL.

## 1. Setup — install dependencies + clone our repo (TB pill dataset)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q vllm==0.6.4 ultralytics fastapi uvicorn python-multipart httpx 2>&1 | tail -3

In [ ]:
# Cloudflared binary — gives free *.trycloudflare.com tunnel
import os
if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
# Clone our repo for TB pill dataset (164 labelled images: rifampicin, isoniazid,
# pyrazinamide, ethambutol, combo_fdc). Public repo, no auth needed.
import os, subprocess
if not os.path.exists('/kaggle/working/davo-ai'):
    !git clone --depth 1 https://github.com/TemurTurayev/davo-ai.git /kaggle/working/davo-ai
!ls /kaggle/working/davo-ai/data/tb_pills_yolo/
print('Train images:', len(os.listdir('/kaggle/working/davo-ai/data/tb_pills_yolo/train/images')))
print('Val images:', len(os.listdir('/kaggle/working/davo-ai/data/tb_pills_yolo/val/images')))

## 2. Retrain YOLO from our preserved dataset (~5-8 min on T4)

We lost the vast.ai-trained weights when the GPU instance was paused, but the
dataset + auto-annotated labels are committed in our repo. So we just retrain.

5 classes: rifampicin · isoniazid · pyrazinamide · ethambutol · combo_fdc

In [ ]:
# Fix dataset.yaml path for Kaggle environment
yaml_path = '/kaggle/working/davo-ai/data/tb_pills_yolo/dataset.yaml'
with open(yaml_path) as f:
    content = f.read()
content = content.replace('/Users/temur/Desktop/Claude/Hackathon_2/davo-ai/', '/kaggle/working/davo-ai/')
with open(yaml_path, 'w') as f:
    f.write(content)
print(content)

In [ ]:
# Train YOLOv8m for 100 epochs on TB pills dataset (~5-8 min on T4)
from ultralytics import YOLO
import torch

model = YOLO('yolov8m.pt')
results = model.train(
    data='/kaggle/working/davo-ai/data/tb_pills_yolo/dataset.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    project='/kaggle/working/runs',
    name='tb_pills_v3',
    exist_ok=True,
    verbose=False,
    patience=20,
    cos_lr=True,
)
best_pt = '/kaggle/working/runs/tb_pills_v3/weights/best.pt'
print(f'\n✅ Trained: {best_pt}')
print(f'mAP@0.5: {results.box.map50:.3f}')
print(f'mAP@0.5:0.95: {results.box.map:.3f}')

## 3. Download Qwen 2.5-7B + Qwen2-VL-7B AWQ models (~10 GB)

In [ ]:
import os
from huggingface_hub import snapshot_download
os.makedirs('/kaggle/working/models', exist_ok=True)

llm_path = snapshot_download(
    repo_id='Qwen/Qwen2.5-7B-Instruct-AWQ',
    local_dir='/kaggle/working/models/llm',
    local_dir_use_symlinks=False,
)
print(f'LLM ready: {llm_path}')

vision_path = snapshot_download(
    repo_id='Qwen/Qwen2-VL-7B-Instruct-AWQ',
    local_dir='/kaggle/working/models/vision',
    local_dir_use_symlinks=False,
)
print(f'Vision ready: {vision_path}')
!du -sh /kaggle/working/models/*

## 4. Start vLLM servers in background

- LLM (port 8001) on GPU 0
- Vision (port 8002) on GPU 1
- YOLO custom-trained (port 8004) on GPU 1 (shares with vision, low VRAM)

In [ ]:
import subprocess, os, time, requests
log_dir = '/kaggle/working/logs'
os.makedirs(log_dir, exist_ok=True)

# LLM on GPU 0
llm_cmd = (
    'CUDA_VISIBLE_DEVICES=0 vllm serve /kaggle/working/models/llm '
    '--served-model-name davoai-llm '
    '--host 0.0.0.0 --port 8001 '
    '--max-model-len 4096 '
    '--gpu-memory-utilization 0.85 '
    '--quantization awq_marlin '
    '--enforce-eager'
)
subprocess.Popen(['bash', '-c', f'{llm_cmd} > {log_dir}/llm.log 2>&1'])
print('🚀 LLM starting on GPU 0 (port 8001)…')

# Vision on GPU 1
vision_cmd = (
    'CUDA_VISIBLE_DEVICES=1 vllm serve /kaggle/working/models/vision '
    '--served-model-name davoai-vision '
    '--host 0.0.0.0 --port 8002 '
    '--max-model-len 4096 '
    '--gpu-memory-utilization 0.75 '
    '--quantization awq_marlin '
    '--limit-mm-per-prompt \'{"image": 2}\' '
    '--enforce-eager'
)
subprocess.Popen(['bash', '-c', f'{vision_cmd} > {log_dir}/vision.log 2>&1'])
print('🚀 Vision starting on GPU 1 (port 8002)…')

In [ ]:
def wait_ready(port, name, timeout=240):
    start = time.time()
    while time.time() - start < timeout:
        try:
            r = requests.get(f'http://localhost:{port}/v1/models', timeout=2)
            if r.status_code == 200:
                print(f'✓ {name} ready ({int(time.time() - start)}s)')
                return True
        except Exception: pass
        time.sleep(3)
    print(f'✗ {name} not ready after {timeout}s — check {log_dir}/{name.lower()}.log')
    return False

wait_ready(8001, 'LLM')
wait_ready(8002, 'Vision')

## 5. YOLO pill detector (port 8004) — uses our retrained TB-specific weights

In [ ]:
yolo_server_code = '''
from fastapi import FastAPI, File, UploadFile
from ultralytics import YOLO
from PIL import Image
import io, time, os

app = FastAPI()
weights = os.environ.get("YOLO_WEIGHTS", "yolov8m.pt")
model = YOLO(weights)
print(f"Loaded YOLO from: {weights}")
print(f"Classes: {model.names}")

@app.get("/health")
async def health():
    return {"status": "ok", "model": weights, "classes": model.names}

@app.post("/detect")
async def detect(image: UploadFile = File(...)):
    t0 = time.time()
    img = Image.open(io.BytesIO(await image.read()))
    results = model(img, verbose=False, conf=0.25)
    detections = []
    for r in results:
        if r.boxes is None: continue
        for b in r.boxes:
            x1, y1, x2, y2 = b.xyxyn[0].tolist()
            detections.append({
                "label": model.names[int(b.cls[0])],
                "confidence": float(b.conf[0]),
                "bbox": [x1, y1, x2, y2],
            })
    return {"detections": detections, "inferenceMs": int((time.time() - t0) * 1000)}
'''
with open('/kaggle/working/yolo_server.py', 'w') as f:
    f.write(yolo_server_code)

subprocess.Popen([
    'bash', '-c',
    f'cd /kaggle/working && YOLO_WEIGHTS={best_pt} CUDA_VISIBLE_DEVICES=1 '
    f'uvicorn yolo_server:app --host 0.0.0.0 --port 8004 > {log_dir}/yolo.log 2>&1',
])
print('🚀 YOLO starting on GPU 1 (port 8004) with our trained weights…')
wait_ready(8004, 'YOLO', timeout=60)

## 6. Multiplex 3 services into one URL via FastAPI proxy (port 7860)

In [ ]:
proxy_code = '''
from fastapi import FastAPI, Request, Response
from fastapi.middleware.cors import CORSMiddleware
import httpx

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
client = httpx.AsyncClient(timeout=120.0)

ROUTES = {
    "/llm": "http://localhost:8001",
    "/vision": "http://localhost:8002",
    "/yolo": "http://localhost:8004",
}

@app.get("/")
async def root():
    return {"davoai": "ok", "routes": list(ROUTES.keys())}

@app.api_route("/{prefix}/{path:path}", methods=["GET", "POST", "PUT", "DELETE", "OPTIONS"])
async def proxy(prefix: str, path: str, request: Request):
    target = ROUTES.get(f"/{prefix}")
    if not target: return Response(content=f"unknown: /{prefix}", status_code=404)
    body = await request.body()
    headers = {k: v for k, v in request.headers.items() if k.lower() not in ("host", "content-length")}
    r = await client.request(request.method, f"{target}/{path}", content=body, headers=headers, params=request.query_params)
    return Response(content=r.content, status_code=r.status_code, headers=dict(r.headers))
'''
with open('/kaggle/working/proxy.py', 'w') as f:
    f.write(proxy_code)

subprocess.Popen([
    'bash', '-c',
    f'cd /kaggle/working && uvicorn proxy:app --host 0.0.0.0 --port 7860 > {log_dir}/proxy.log 2>&1',
])
time.sleep(3)
print(requests.get('http://localhost:7860/').json())

## 7. Open Cloudflare Tunnel — get public URL

In [ ]:
import re, os
from kaggle_secrets import UserSecretsClient

# Try to load named-tunnel token from Kaggle Secrets (Add-ons -> Secrets).
# If present -> stable hostname configured in Cloudflare Dashboard.
# Else -> random *.trycloudflare.com (changes each session).
tunnel_url = None
token = None
try:
    token = UserSecretsClient().get_secret("CLOUDFLARE_TUNNEL_TOKEN")
    print("✓ Named tunnel token found in Kaggle Secrets")
except Exception as e:
    print(f"No named-tunnel token (using random): {e}")

if token:
    # Named tunnel — Cloudflare dashboard owns the routing config.
    # Make sure the Public Hostname in Cloudflare points to localhost:7860.
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "run", "--token", token],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1, universal_newlines=True,
    )
    print("Named tunnel connector starting...")
    # Try to extract hostname from logs
    for _ in range(40):
        line = proc.stdout.readline()
        if not line: time.sleep(0.5); continue
        print(line.rstrip())
        # Cloudflared prints "Registered tunnel connection" when up
        m = re.search(r"https?://([a-zA-Z0-9.\-]+\.[a-z]{2,})", line)
        if m and "trycloudflare" not in m.group(0):
            tunnel_url = m.group(0)
            break
        if "Registered tunnel connection" in line:
            print("\n⚠️  Named tunnel is up. Hostname is configured in Cloudflare Dashboard,")
            print("    not visible in logs. Check Cloudflare Dashboard -> Tunnels -> Public Hostnames.")
            tunnel_url = "<set in Cloudflare Dashboard>"
            break
else:
    # Random trycloudflare.com fallback
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:7860", "--no-autoupdate"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1, universal_newlines=True,
    )
    for _ in range(60):
        line = proc.stdout.readline()
        if not line: time.sleep(0.5); continue
        print(line.rstrip())
        m = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", line)
        if m:
            tunnel_url = m.group(0)
            break

print("\n" + "=" * 70)
if tunnel_url:
    print(f"✅ PUBLIC URL: {tunnel_url}")
    print("\nAdd to your local .env.local:")
    print(f"NEXT_PUBLIC_KAGGLE_INFERENCE_URL={tunnel_url}")
    print("\nEndpoints:")
    print(f"  {tunnel_url}/llm/v1/chat/completions")
    print(f"  {tunnel_url}/vision/v1/chat/completions")
    print(f"  {tunnel_url}/yolo/detect")
    # Auto-publish to ntfy for Mac listener
    try:
        ntfy_resp = requests.post(
            "https://ntfy.sh/davoai-tunnel-temur1210",
            data=tunnel_url.encode("utf-8"),
            headers={"Title": "TB Control tunnel ready", "Tags": "rocket"},
            timeout=10,
        )
        print(f"\n📡 Published URL to ntfy.sh/davoai-tunnel-temur1210 (status {ntfy_resp.status_code})")
    except Exception as e:
        print(f"\n⚠️  Could not publish to ntfy: {e}")
else:
    print("✗ Tunnel did not come up")
print("=" * 70)


## 8. Keep-alive loop (prevent 20-min idle kill)

**Run this cell and leave it running** for the entire session. Pings every 12 min.

In [ ]:
from datetime import datetime
while True:
    try:
        r = requests.get('http://localhost:7860/', timeout=5)
        print(f'[{datetime.now().strftime("%H:%M:%S")}] alive · {r.status_code}')
    except Exception as e:
        print(f'[{datetime.now().strftime("%H:%M:%S")}] err: {e}')
    time.sleep(60 * 12)